# ClusterAPI

---

Installation CLI

In [ ]:
%%bash
curl -L https://github.com/kubernetes-sigs/cluster-api/releases/download/v1.13.3/clusterctl-linux-amd64 -o clusterctl
sudo install -o root -g root -m 0755 clusterctl /usr/local/bin/clusterctl

### Load Balancer



In [ ]:
%%bash
METALLB_VER=$(curl "https://api.github.com/repos/metallb/metallb/releases/latest" | jq -r ".tag_name")
kubectl apply -f "https://raw.githubusercontent.com/metallb/metallb/${METALLB_VER}/config/manifests/metallb-native.yaml"
kubectl wait pods -n metallb-system -l app=metallb,component=controller --for=condition=Ready --timeout=10m
kubectl wait pods -n metallb-system -l app=metallb,component=speaker --for=condition=Ready --timeout=2m
cat <<EOF | kubectl apply -f -
apiVersion: metallb.io/v1beta1
kind: IPAddressPool
metadata:
  name: capi-ip-pool
  namespace: metallb-system
spec:
  addresses:
  - 10.10.0.30-10.10.0.90
---
apiVersion: metallb.io/v1beta1
kind: L2Advertisement
metadata:
  name: empty
  namespace: metallb-system
EOF


### Cluster Initialisieren

Initialisieren Sie den Management-Cluster mit dem KubeVirt-Provider.

In [ ]:
%%bash
clusterctl init --infrastructure kubevirt

export NODE_VM_IMAGE_TEMPLATE="quay.io/capk/ubuntu-2404-container-disk:v1.32.1"
export CAPK_GUEST_K8S_VERSION="${NODE_VM_IMAGE_TEMPLATE/*:/}"
export CRI_PATH="unix:///var/run/containerd/containerd.sock"
    
clusterctl generate cluster capi-quickstart \
  --infrastructure="kubevirt" \
  --flavor lb \
  --kubernetes-version ${CAPK_GUEST_K8S_VERSION} \
  --control-plane-machine-count=1 \
  --worker-machine-count=2 \
  > capi-quickstart.yaml


In [ ]:
%%bash
kubectl apply -f capi-quickstart.yaml

### Hack wegen fehlendem Containerd Path im Template bei den Workern

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: bootstrap.cluster.x-k8s.io/v1beta1
kind: KubeadmConfigTemplate
metadata:
  name: capi-quickstart-md-0
  namespace: default
spec:
  template:
    spec:
      joinConfiguration:
        nodeRegistration:
          criSocket: unix:///var/run/containerd/containerd.sock
          kubeletExtraArgs: {}
EOF

Anzeige der erstellen Ressourcen.

Wir müssen warten bis die kubeadmcontrolplane bereit ist, für die nächsten Schritte

In [ ]:
%%bash
kubectl get vm,vmi,cluster,kubeadmcontrolplane

Nachdem kubeadmcontrolplane bereit ist, können wir kubeconfig erzeugen

In [ ]:
%%bash
clusterctl get kubeconfig capi-quickstart > capi-quickstart.kubeconfig
kubectl --kubeconfig capi-quickstart.kubeconfig get pods -A


### Overlay Netzwerk

In [ ]:
%%bash
curl https://raw.githubusercontent.com/projectcalico/calico/v3.29.1/manifests/calico.yaml -o calico-workload.yaml

sed -i -E 's|^( +)# (- name: CALICO_IPV4POOL_CIDR)$|\1\2|g;'\
's|^( +)# (  value: )"192.168.0.0/16"|\1\2"10.243.0.0/16"|g;'\
'/- name: CLUSTER_TYPE/{ n; s/( +value: ").+/\1k8s"/g };'\
'/- name: CALICO_IPV4POOL_IPIP/{ n; s/value: "Always"/value: "Never"/ };'\
'/- name: CALICO_IPV4POOL_VXLAN/{ n; s/value: "Never"/value: "Always"/};'\
'/# Set Felix endpoint to host default action to ACCEPT./a\            - name: FELIX_VXLANPORT\n              value: "6789"' \
calico-workload.yaml
kubectl --kubeconfig=./capi-quickstart.kubeconfig create -f calico-workload.yaml


In [ ]:
%%bash
export KUBECONFIG=`pwd`/capi-quickstart.kubeconfig
kubectl get nodes
kubectl get pods -A -o wide

### Headlamp starten

In [ ]:
%%bash
export KUBECONFIG=`pwd`/capi-quickstart.kubeconfig
kubectl apply -f https://raw.githubusercontent.com/headlamp-k8s/headlamp/main/kubernetes-headlamp.yaml
kubectl -n kube-system create serviceaccount headlamp-admin
kubectl create clusterrolebinding headlamp-admin --serviceaccount=kube-system:headlamp-admin --clusterrole=cluster-admin

kubectl patch svc headlamp \
  -n kube-system \
  --type='merge' \
  -p '{
    "spec": {
      "type": "NodePort",
      "ports": [
        {
          "port": 80,
          "targetPort": 4466,
          "nodePort": 30444
        }
      ]
    }
  }'
echo "HeadLamp: http://"10.10.0.31":30444"
kubectl create token headlamp-admin -n kube-system --duration=48h